# Datasets and DataLoaders

`Dataset` holds the samples and their labels and answers two questions: how many samples are there, and give me sample number `i`.

`DataLoader` wraps a `Dataset` and handles batching, shuffling and parallel loading.

Keeping the two apart means the data code stays separate from the training code.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
from torchvision.io import read_image
from torchvision.transforms import ToTensor

## A built-in dataset

FashionMNIST holds 28x28 grayscale clothing images in 10 classes. The first run downloads about 30 MB into `data/`, which is ignored by git.

`transform=ToTensor()` turns the PIL image into a float tensor and scales the pixels from 0-255 into 0.0-1.0.

In [ ]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

print(len(training_data))
print(len(test_data))

Indexing a `Dataset` returns one pair, not a batch. The image is `[channels, height, width]` and the label is a plain int.

In [ ]:
img, label = training_data[0]

print(img.shape)
print(img.dtype)
print(img.min().item(), img.max().item())
print(label, type(label))

## Looking at the data

`imshow` wants `[height, width]` for a grayscale image, so the channel dimension has to go. `squeeze()` drops dimensions of size 1, turning `[1, 28, 28]` into `[28, 28]`.

In [ ]:
labels_map = {
    0: "T-Shirt",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle Boot",
}

figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3

for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(training_data), size=(1,)).item()
    img, label = training_data[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(labels_map[label])
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")

plt.show()

## A custom Dataset

A custom dataset needs exactly three methods:

- `__init__` runs once and stores paths, the label table and the transforms
- `__len__` returns the number of samples
- `__getitem__` loads and returns one sample by index

Load images inside `__getitem__`, not in `__init__`. Reading everything up front wastes memory once the dataset is bigger than RAM.

First some small files to load, so the class below has something real to read.

In [ ]:
example_dir = Path("data/custom_example")
image_dir = example_dir / "images"
image_dir.mkdir(parents=True, exist_ok=True)

rows = []
for i in range(6):
    name = f"img_{i}.png"
    array = (torch.rand(28, 28) * 255).to(torch.uint8).numpy()
    Image.fromarray(array, mode="L").save(image_dir / name)
    rows.append([name, i % 3])

with open(example_dir / "labels.csv", "w", newline="") as f:
    csv.writer(f).writerows(rows)

print(sorted(p.name for p in image_dir.iterdir()))

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(
        self, annotations_file, img_dir, transform=None, target_transform=None
    ):
        self.img_labels = pd.read_csv(annotations_file, header=None)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = self.img_dir / self.img_labels.iloc[idx, 0]
        image = read_image(str(img_path))
        label = self.img_labels.iloc[idx, 1]

        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)

        return image, label

`read_image` returns `uint8` in the range 0-255, not the 0.0-1.0 floats that `ToTensor()` produced above. Feeding `uint8` straight into a layer raises a dtype error, so a transform has to convert it.

In [ ]:
custom_data = CustomImageDataset(example_dir / "labels.csv", image_dir)

image, label = custom_data[0]

print(len(custom_data))
print(image.shape, image.dtype)
print(image.min().item(), image.max().item())
print(label)

## DataLoader

A `Dataset` gives one sample at a time. Training needs batches, reshuffled every epoch so the model does not learn the order.

`len(dataloader)` is the number of batches, not the number of samples. 60000 samples with `batch_size=64` gives 938 batches, the last one being smaller.

Shuffle the training set, not the test set. Test results should not depend on the order.

In [ ]:
train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=False)

print(len(train_dataloader), len(train_dataloader.dataset))
print(len(test_dataloader), len(test_dataloader.dataset))

The loader stacks samples into one tensor and adds a batch dimension in front:

```text
one sample:  [1, 28, 28]
one batch:   [64, 1, 28, 28]
labels:      [64]
```

In [ ]:
train_features, train_labels = next(iter(train_dataloader))

print(train_features.shape)
print(train_labels.shape)
print(train_features.dtype, train_labels.dtype)

In [ ]:
img = train_features[0].squeeze()
label = train_labels[0].item()

plt.imshow(img, cmap="gray")
plt.axis("off")
plt.show()

print(labels_map[label])

A normal training loop iterates the loader directly, one pass being one epoch:

```python
for batch, (X, y) in enumerate(train_dataloader):
    ...
```

On this Mac leave `num_workers` at 0 to start. Worker processes help with slow disk reads, but they add startup cost and can hang in notebooks.